# Sentient AI — Milestone 1 Resubmission Notebook
INAI 6986 · Foluwasade Ajagbe

This notebook rebuilds the data preparation, baseline, comparator, and candidate models
from the raw MTSamples source, and adds the previously-missing LLM (Claude API) candidate.

**Note on provenance:** the original `train.csv`/`validation.csv` from the first Milestone 1
submission were not recovered, so this notebook regenerates the split from
`mtsamples-v3.csv` with a fresh, fully-documented set of rules below. Every number in
this notebook is produced by the code in this notebook — nothing is pasted in.

In [ ]:
import hashlib, re, json, warnings
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

SRC = "mtsamples-v3.csv"   # place the raw source file next to this notebook
SEED = 42

## 1. Source hashing and load

In [ ]:
with open(SRC, "rb") as f:
    src_hash = hashlib.sha256(f.read()).hexdigest()
print("SHA-256 of source:", src_hash)

df = pd.read_csv(SRC, on_bad_lines='skip').rename(columns={"Unnamed: 0": "row_id"})
for col in ["medical_specialty", "sample_name", "transcription", "keywords", "description"]:
    df[col] = df[col].astype(str).str.strip()
print("Raw rows:", len(df))

SHA-256 of source: baa6d482c2d25362f6e082fda5756a8faaef805e2cf6b03fd5c6f7946588b272
Raw rows: 4999


## 2. Class selection

**Rule:** keep single-specialty outpatient referral targets — the kind of specialty a
GP would route a non-emergency patient to. Exclude document-type / encounter-type
categories (`Surgery`, `Consult - History and Phy.`, `SOAP / Chart / Progress Notes`,
`Discharge Summary`, `Emergency Room Reports`, `Office Notes`, `Letters`, `Autopsy`,
`General Medicine`) because those describe *where a note was written*, not a specialty
a referral engine would route to.

In [ ]:
#Target class and select a specialist
TARGET_CLASSES = [
    "Cardiovascular / Pulmonary", "Orthopedic", "Gastroenterology", "Neurology",
    "Obstetrics / Gynecology", "Urology", "ENT - Otolaryngology", "Nephrology",
    "Dermatology",
]
sub = df[df["medical_specialty"].isin(TARGET_CLASSES)].copy()
sub = sub[~sub["transcription"].isin(["", "nan"])].copy()
print("After class selection + drop-empty:", len(sub))

After class selection + drop-empty: 1690


## 3. Exact-duplicate removal

In [ ]:
#Remove the empty duplicate before splitting
before = len(sub)
sub = sub.drop_duplicates(subset="transcription", keep="first").copy()
print(f"After exact-duplicate removal: {len(sub)} (dropped {before - len(sub)})")

After exact-duplicate removal: 1610 (dropped 80)


## 4. Multi-label removal

**Rule:** a note is flagged multi-label if its `keywords` field name-checks one of the
*other* 8 target specialties (lowercased substring match on each specialty's name
tokens). This catches notes MTSamples' own indexers associated with more than one
target specialty.

In [ ]:
def specialty_tokens(name):
    parts = re.split(r"[\/\-]", name)
    return [p.strip().lower() for p in parts if len(p.strip()) > 3]

spec_tokens = {c: specialty_tokens(c) for c in TARGET_CLASSES}

def is_multilabel(row):
    kw = str(row["keywords"]).lower()
    own = row["medical_specialty"]
    return any(any(tok in kw for tok in toks) for other, toks in spec_tokens.items() if other != own)

sub["multilabel_flag"] = sub.apply(is_multilabel, axis=1)
before = len(sub)
sub = sub[~sub["multilabel_flag"]].copy()
print(f"After multi-label removal: {len(sub)} (dropped {before - len(sub)})")
print(sub["medical_specialty"].value_counts())

After multi-label removal: 1470 (dropped 140)
medical_specialty
Cardiovascular / Pulmonary    359
Orthopedic                    353
Gastroenterology              203
Urology                       156
Obstetrics / Gynecology       154
ENT - Otolaryngology           95
Nephrology                     67
Neurology                      54
Dermatology                    29
Name: count, dtype: int64


## 5. Pre-decision truncation (the leakage fix)

Truncate every note at the **earliest** occurrence of a post-decision section header
(`PREOPERATIVE DIAGNOSIS`, `IMPRESSION`, `ASSESSMENT`, `PLAN`, `PROCEDURE`, etc.).
Notes with less than 40 characters of pre-decision text remaining are dropped — mostly
short operative notes that open directly with `PREOPERATIVE DIAGNOSIS:` and have no
usable referral-style narrative before it.

In [ ]:
DECISION_HEADERS = [
    "PREOPERATIVE DIAGNOSIS", "PREOPERATIVE DIAGNOSES", "POSTOPERATIVE DIAGNOSIS",
    "POSTOPERATIVE DIAGNOSES", "PROCEDURE PERFORMED", "PROCEDURE IN DETAIL",
    "DESCRIPTION OF PROCEDURE", "PROCEDURE", "PROCEDURES", "ANESTHESIA", "FINDINGS",
    "IMPRESSION", "ASSESSMENT AND PLAN", "ASSESSMENT", "DIAGNOSIS", "DIAGNOSES", "PLAN",
    "COMPLICATIONS", "ESTIMATED BLOOD LOSS", "SPECIMENS", "INDICATIONS", "INDICATION",
    "DISPOSITION", "RECOMMENDATION", "RECOMMENDATIONS",
]
pattern = re.compile(
    r"(?:^|[\.,\n])\s*(" + "|".join(re.escape(h) for h in DECISION_HEADERS) + r")\s*:",
    re.IGNORECASE,
)

def truncate_pre_decision(text):
    text = str(text)
    m = pattern.search(text)
    if m:
        return text[:m.start()].strip(), True, m.group(1).upper()
    return text.strip(), False, None

trunc_results = sub["transcription"].apply(truncate_pre_decision)
sub["text"] = trunc_results.apply(lambda x: x[0])
sub["truncated"] = trunc_results.apply(lambda x: x[1])
sub["cut_header"] = trunc_results.apply(lambda x: x[2])
sub["text_len"] = sub["text"].str.len()

print("Truncated (decision header found):", sub["truncated"].sum(), "/", len(sub))

before = len(sub)
sub = sub[sub["text_len"] >= 40].copy()
print(f"After dropping too-short pre-decision text: {len(sub)} (dropped {before - len(sub)})")
print(sub["medical_specialty"].value_counts())

Truncated (decision header found): 1236 / 1470
After dropping too-short pre-decision text: 741 (dropped 729)
medical_specialty
Cardiovascular / Pulmonary    230
Orthopedic                    147
Gastroenterology               78
Obstetrics / Gynecology        70
Urology                        59
Nephrology                     52
Neurology                      51
ENT - Otolaryngology           37
Dermatology                    17
Name: count, dtype: int64


## 6. Near-duplicate check

Cosine similarity on TF-IDF vectors of the *pre-decision* text, between all pairs of
remaining notes. Pairs at or above 0.9 are grouped with union-find so a near-duplicate
pair can never be split across train and validation.

In [ ]:
sub = sub.reset_index(drop=True)
sub["case_id"] = ["case_%04d" % i for i in range(len(sub))]

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=1)
X = vec.fit_transform(sub["text"])
sim = cosine_similarity(X)
np.fill_diagonal(sim, 0)

THRESH = 0.9
pairs = np.argwhere(sim >= THRESH)
pairs = pairs[pairs[:, 0] < pairs[:, 1]]
print(f"Near-duplicate pairs (cosine >= {THRESH}): {len(pairs)} out of {len(sub)} notes")

parent = list(range(len(sub)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb
for a, b in pairs:
    union(int(a), int(b))
sub["dup_group"] = [find(i) for i in range(len(sub))]
print("Groups after clustering:", sub["dup_group"].nunique(), "(from", len(sub), "notes)")

Near-duplicate pairs (cosine >= 0.9): 2 out of 741 notes
Groups after clustering: 739 (from 741 notes)


## 7. Stratified, group-aware train/validation split

17% validation, stratified by specialty, with near-duplicate groups kept intact.

In [ ]:
group_rep = sub.groupby("dup_group").first().reset_index()
train_groups, val_groups = train_test_split(
    group_rep["dup_group"], test_size=0.17,
    stratify=group_rep["medical_specialty"], random_state=SEED,
)
train_df = sub[sub["dup_group"].isin(train_groups)].copy()
val_df = sub[sub["dup_group"].isin(val_groups)].copy()

out_cols = ["case_id", "medical_specialty", "text", "sample_name", "truncated", "cut_header"]
train = train_df[out_cols].rename(columns={"medical_specialty": "specialty"})
val = val_df[out_cols].rename(columns={"medical_specialty": "specialty"})

assert set(train.case_id).isdisjoint(set(val.case_id))
assert train.text.notna().all() and val.text.notna().all()

train.to_csv("train.csv", index=False)
val.to_csv("validation.csv", index=False)

print("Train:", len(train), " Validation:", len(val))
print("\nTrain class counts:\n", train["specialty"].value_counts())
print("\nValidation class counts:\n", val["specialty"].value_counts())
labels = sorted(train["specialty"].unique().tolist())

Train: 614  Validation: 127

Train class counts:
 specialty
Cardiovascular / Pulmonary    191
Orthopedic                    121
Gastroenterology               65
Obstetrics / Gynecology        58
Urology                        49
Nephrology                     43
Neurology                      42
ENT - Otolaryngology           31
Dermatology                    14
Name: count, dtype: int64

Validation class counts:
 specialty
Cardiovascular / Pulmonary    39
Orthopedic                    26
Gastroenterology              13
Obstetrics / Gynecology       12
Urology                       10
Neurology                      9
Nephrology                     9
ENT - Otolaryngology           6
Dermatology                    3
Name: count, dtype: int64


## 8. Leakage check: truncated vs. untruncated text

Same split, same model, run once on the pre-decision text and once on the full
(untruncated) transcription, to show what the post-decision sections were worth.

In [ ]:
full_lookup = sub.set_index("case_id")["transcription"]
train_full = train["case_id"].map(full_lookup)
val_full = val["case_id"].map(full_lookup)

tfidf_leak = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
Xtr_full = tfidf_leak.fit_transform(train_full)
Xval_full = tfidf_leak.transform(val_full)
clf_leak = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr_full, train["specialty"])
pred_leak = clf_leak.predict(Xval_full)

acc_leak = accuracy_score(val["specialty"], pred_leak)
f1_leak = f1_score(val["specialty"], pred_leak, average="macro", zero_division=0)
print(f"Logistic Regression on UNTRUNCATED text: accuracy={acc_leak:.3f}  macro F1={f1_leak:.3f}")
print("(kept here to demonstrate the leakage effect — the truncated-text score below is the real one)")

Logistic Regression on UNTRUNCATED text: accuracy=0.890  macro F1=0.900
(kept here to demonstrate the leakage effect — the truncated-text score below is the real one)


## 9. Keyword baseline

**Provenance:** every specialty's keyword list was written by reading a sample of
**training** notes only for that specialty. No validation note was consulted while
writing this list.

In [ ]:
KEYWORDS = {
    "Cardiovascular / Pulmonary": ["heart", "cardiac", "coronary", "pulmonary", "lung", "chest pain", "ejection fraction", "artery"],
    "Orthopedic": ["fracture", "joint", "knee", "hip", "shoulder", "orthopedic", "bone", "ligament"],
    "Gastroenterology": ["abdomen", "abdominal", "colon", "bowel", "gastric", "liver", "gi ", "stomach"],
    "Obstetrics / Gynecology": ["pregnan", "uterus", "cervix", "ovary", "vaginal", "gestational", "fetal"],
    "Urology": ["bladder", "prostate", "urinary", "kidney stone", "urology", "urethra", "scrotal"],
    "Nephrology": ["renal", "kidney", "dialysis", "nephro", "creatinine", "glomerular"],
    "Neurology": ["seizure", "neurolog", "headache", "brain", "cranial", "stroke", "weakness"],
    "ENT - Otolaryngology": ["throat", "ear", "nasal", "sinus", "tonsil", "hearing", "larynx"],
    "Dermatology": ["skin", "rash", "lesion", "dermat", "mole", "biopsy of the skin"],
}
def keyword_predict(text):
    t = str(text).lower()
    scores = {c: sum(t.count(k) for k in kws) for c, kws in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Cardiovascular / Pulmonary"

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    print(f"\n=== {name} ===\nTop-1 accuracy: {acc:.3f}   Macro F1: {f1:.3f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    return acc, f1

dummy = DummyClassifier(strategy="most_frequent").fit(train["text"], train["specialty"])
report("Majority-class sanity check", val["specialty"], dummy.predict(val["text"]))

kw_pred = val["text"].apply(keyword_predict)
report("Keyword baseline", val["specialty"], kw_pred)


=== Majority-class sanity check ===
Top-1 accuracy: 0.307   Macro F1: 0.052
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.31      1.00      0.47        39
               Dermatology       0.00      0.00      0.00         3
      ENT - Otolaryngology       0.00      0.00      0.00         6
          Gastroenterology       0.00      0.00      0.00        13
                Nephrology       0.00      0.00      0.00         9
                 Neurology       0.00      0.00      0.00         9
   Obstetrics / Gynecology       0.00      0.00      0.00        12
                Orthopedic       0.00      0.00      0.00        26
                   Urology       0.00      0.00      0.00        10

                  accuracy                           0.31       127
                 macro avg       0.03      0.11      0.05       127
              weighted avg       0.09      0.31      0.14       127


=== Keyword baseline ===
Top-1 accu

(0.5354330708661418, 0.4866613758355307)

## 10. Non-DL comparator: Logistic Regression on TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
Xtr = tfidf.fit_transform(train["text"])
Xval = tfidf.transform(val["text"])

logreg = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train["specialty"])
report("Logistic Regression (TF-IDF)", val["specialty"], logreg.predict(Xval))


=== Logistic Regression (TF-IDF) ===
Top-1 accuracy: 0.811   Macro F1: 0.814
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.91      0.79      0.85        39
               Dermatology       1.00      1.00      1.00         3
      ENT - Otolaryngology       1.00      0.50      0.67         6
          Gastroenterology       0.50      0.69      0.58        13
                Nephrology       0.69      1.00      0.82         9
                 Neurology       0.67      0.89      0.76         9
   Obstetrics / Gynecology       1.00      0.83      0.91        12
                Orthopedic       0.88      0.81      0.84        26
                   Urology       0.90      0.90      0.90        10

                  accuracy                           0.81       127
                 macro avg       0.84      0.82      0.81       127
              weighted avg       0.84      0.81      0.82       127



(0.8110236220472441, 0.8139782650697366)

## 11. MLP candidates — fixed

Previous run showed MLP 64-32 collapsing to near-majority-class behavior. Fix: explicit
`max_iter=3000`, `early_stopping=False` (401-row train set is too small to spend 10% of
it on an internal holdout), and convergence is checked and printed, not assumed.

In [ ]:
def fit_mlp(hidden, tag):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        clf = MLPClassifier(hidden_layer_sizes=hidden, max_iter=3000,
                             early_stopping=False, random_state=SEED, alpha=1e-3)
        clf.fit(Xtr, train["specialty"])
        conv_warnings = [str(x.message) for x in w if "converge" in str(x.message).lower()]
    print(f"[{tag}] iterations run: {clf.n_iter_}  final loss: {clf.loss_:.4f}  convergence warnings: {len(conv_warnings)}")
    report(f"MLP {tag} (TF-IDF)", val["specialty"], clf.predict(Xval))
    return clf

mlp_64 = fit_mlp((64,), "64")
mlp_64_32 = fit_mlp((64, 32), "64-32")

[64] iterations run: 204  final loss: 0.0151  convergence warnings: 0

=== MLP 64 (TF-IDF) ===
Top-1 accuracy: 0.850   Macro F1: 0.812
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.83      0.87      0.85        39
               Dermatology       1.00      0.33      0.50         3
      ENT - Otolaryngology       1.00      0.67      0.80         6
          Gastroenterology       0.69      0.69      0.69        13
                Nephrology       0.89      0.89      0.89         9
                 Neurology       0.89      0.89      0.89         9
   Obstetrics / Gynecology       0.83      0.83      0.83        12
                Orthopedic       0.86      0.96      0.91        26
                   Urology       1.00      0.90      0.95        10

                  accuracy                           0.85       127
                 macro avg       0.89      0.78      0.81       127
              weighted avg       0.86      0.85

## 12. LLM candidate — Claude API (previously missing)

This is the declared model family and was never tested in the first submission.
Zero-shot, same 127 validation cases, same pre-decision-truncated text as every other
candidate above. **Requires `ANTHROPIC_API_KEY` in the environment — not executed in
this development run; run this cell yourself before submitting.**

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("New Anthropic API key:")
if not RUN_LLM:
    print("ANTHROPIC_API_KEY not set - skipping live call. "
          "Set the key and re-run this cell before submitting.")
else:
    from anthropic import Anthropic
    client = Anthropic()
    model_id = "claude-sonnet-5"
    prompt_version = "zero_shot_v1"

    schema = {
        "type": "object",
        "properties": {"specialty": {"type": "string", "enum": labels}},
        "required": ["specialty"],
        "additionalProperties": False,
    }
    system_prompt = (
        "Classify a de-identified clinical note for a nonemergency outpatient "
        "referral. Choose exactly one specialty from the allowed list. "
        "Use only the supplied pre-decision text. Do not infer an answer from "
        "missing assessment, diagnosis, procedure, or referral sections. "
        "This is a research classification task, not a clinical decision."
    )

    results = []
    for row in val.itertuples(index=False):
        response = client.messages.create(
            model=model_id,
            max_tokens=128,
            system=system_prompt,
            messages=[{
                "role": "user",
                "content": f"Allowed specialties: {', '.join(labels)}\n\nPre-decision note:\n{row.text}",
            }],
            output_config={"format": {"type": "json_schema", "schema": schema}},
        )
        block = next(b for b in response.content if b.type == "text")
        prediction = json.loads(block.text)["specialty"]
        results.append({"case_id": row.case_id, "gold": row.specialty,
                         "prediction": prediction, "model_id": model_id,
                         "prompt_version": prompt_version})

    out = pd.DataFrame(results)
    out.to_csv("claude_validation_predictions.csv", index=False)
    report("Claude (zero-shot)", out.gold, out.prediction)

New Anthropic API key:··········
ANTHROPIC_API_KEY not set - skipping live call. Set the key and re-run this cell before submitting.


## 13. Per-class recall with raw counts

Percentages alone hide sample size on small classes (Dermatology has 3 validation
cases, Nephrology 9). Report both.

In [ ]:
from sklearn.metrics import recall_score

final_model_pred = mlp_64.predict(Xval)  # swap in whichever candidate is selected
for cls in labels:
    n = (val["specialty"] == cls).sum()
    correct = ((val["specialty"] == cls) & (pd.Series(final_model_pred, index=val.index) == cls)).sum()
    print(f"{cls:30s}: {correct}/{n} correct")

Cardiovascular / Pulmonary    : 34/39 correct
Dermatology                   : 1/3 correct
ENT - Otolaryngology          : 4/6 correct
Gastroenterology              : 9/13 correct
Nephrology                    : 8/9 correct
Neurology                     : 8/9 correct
Obstetrics / Gynecology       : 10/12 correct
Orthopedic                    : 25/26 correct
Urology                       : 9/10 correct


## 14. Interpretability — global view + occlusion

Top predictive terms per class (global), plus one occlusion example on a correctly
classified case (removing that case's top-weighted words and observing the probability
drop for the true class).

In [ ]:
feat_names = np.array(tfidf.get_feature_names_out())
print("=== Top 6 terms per class (Logistic Regression coefficients) ===")
for i, cls in enumerate(logreg.classes_):
    top_idx = np.argsort(logreg.coef_[i])[-6:][::-1]
    print(f"{cls:30s}: {', '.join(feat_names[top_idx])}")

pred_lr = logreg.predict(Xval)
correct_mask = pred_lr == val["specialty"].values
idx = int(np.argmax(correct_mask))
row = val.iloc[idx]
true_cls = row["specialty"]
cls_idx = list(logreg.classes_).index(true_cls)
base_proba = logreg.predict_proba(Xval[idx])[0][cls_idx]

top_words = feat_names[np.argsort(logreg.coef_[cls_idx])[-5:]]
occluded_text = str(row["text"])
for w in top_words:
    occluded_text = occluded_text.replace(w, "")
Xocc = tfidf.transform([occluded_text])
occ_proba = logreg.predict_proba(Xocc)[0][cls_idx]

print(f"\nCase {row['case_id']}  true={true_cls}")
print(f"P(true class) before occlusion: {base_proba:.3f}")
print(f"Top words removed: {list(top_words)}")
print(f"P(true class) after occlusion:  {occ_proba:.3f}  (drop of {base_proba - occ_proba:.3f})")

=== Top 6 terms per class (Logistic Regression coefficients) ===
Cardiovascular / Pulmonary    : chest, coronary, artery, pulmonary, heart, cardiac
Dermatology                   : acne, buttock, rash, skin, abscess, size cm
ENT - Otolaryngology          : ear, nasal, bilateral, tonsils, eardrum, nose
Gastroenterology              : colon, bowel, procedure, colonoscopy, liver, abdominal
Nephrology                    : renal, kidney, pelvis, ct abdomen, ct, renal disease
Neurology                     : motor, reported, left, right, weakness, husband
Obstetrics / Gynecology       : fetal, pregnancy, breast, menorrhagia, vaginal, pelvic
Orthopedic                    : pain, spine, fracture, shoulder, knee, mri
Urology                       : prostate, psa, bladder, penis, hematuria, urinary

Case case_0001  true=Cardiovascular / Pulmonary
P(true class) before occlusion: 0.543
Top words removed: ['heart', 'pulmonary', 'artery', 'coronary', 'chest']
P(true class) after occlusion:  0.485  (dr

## 15. Summary table

In [ ]:
summary_rows = []
def add_row(name, y_pred):
    acc = accuracy_score(val["specialty"], y_pred)
    f1 = f1_score(val["specialty"], y_pred, labels=labels, average="macro", zero_division=0)
    summary_rows.append({"model": name, "top1_accuracy": round(acc,3), "macro_f1": round(f1,3)})

add_row("Majority class", dummy.predict(val["text"]))
add_row("Keyword baseline", kw_pred)
add_row("Logistic Regression (TF-IDF)", logreg.predict(Xval))
add_row("MLP (64)", mlp_64.predict(Xval))
add_row("MLP (64-32)", mlp_64_32.predict(Xval))
if RUN_LLM:
    add_row("Claude (zero-shot)", out.prediction)

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary.to_csv("week7_summary.csv", index=False)

                       model  top1_accuracy  macro_f1
              Majority class          0.307     0.052
            Keyword baseline          0.535     0.487
Logistic Regression (TF-IDF)          0.811     0.814
                    MLP (64)          0.850     0.812
                 MLP (64-32)          0.811     0.763


 **Intrepret**

Logistic Regression(81.1% accuracy) and neural network moesl significantly outperform both the majority class (30.7%) and Keyword baselines (53.5%). This confirms that statisical n-gram representation is necessary to capture non-exact clinical terminology and context.
**Deep Learning Performance: ** The MLP (64) model achieves the top performance overall with 85.0% Top-1 Accuracy and an 0.812 Macro F1 score.